# Vanilla Agentic Search

This notebook walks you through a vanilla agentic search example with a search tool to run semantic search queries over a database.

<img src="../img/Vanilla Agentic Search.png" width="1000" alt="Vanilla Agentic search (Agentic RAG)" />

## Preparation

Make sure you have completed the setup instructions according to the README.md.

Load environment variables from `.env` file, such as API keys.

In [1]:
import os                                                                                                                                                                                                          
from dotenv import load_dotenv

load_dotenv()

LITELLM_API_KEY=os.getenv("LITELLM_API_KEY")
LITELLM_API_BASE=os.getenv("LITELLM_API_BASE")

ES_URL=os.getenv("ELASTICSEARCH_URL")
ES_USERNAME=os.getenv("ELASTICSEARCH_USERNAME")
ES_PASSWORD=os.getenv("ELASTICSEARCH_PASSWORD")

JINA_API_KEY=os.getenv("JINA_API_KEY")

## Set up basic agent with a vector search tool

Follows the [basic Quickstart](https://docs.langchain.com/oss/python/langchain/quickstart)

### Configure the LLM

Set up your LLM with the right parameters for your use case:
Below, we're using OpenAI's models through LiteLLM but you can switch it out with any model capable of tool use.

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_base=LITELLM_API_BASE,
    api_key=LITELLM_API_KEY,
    model="llm-gateway/gpt-5.4-nano",
    temperature=0.5,
)

/Users/leonie/Documents/code/workshop-agentic-search/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


### Define the system prompt

The system prompt defines your agent’s role and behavior. 

We also include information about what out-of-context knowledge source it has. In this case it has access to an **Elasticsearch index** with the conference data.

In [3]:
from IPython.display import display, Markdown
SYSTEM_PROMPT = open("system_prompt_db.md", 'r').read()

display(Markdown(SYSTEM_PROMPT))

You are a search agent tasked with answering questions about the AI Engineer Europe 2026 Conference.

You have access to different context retrieval tools to help you answer user queries.

Before answering a question decide whether or not you need to retrieve additional context to answer the question correctly.
If the retrieved context does not contain relevant information to answer the query, say that you don't know. 

## Elasticsearch (`conference_schedule`)

The conference sessions are indexed in Elasticsearch. One document per session.

| Field | Description |
|--------------|------------|
| `text` | The string that was embedded: each session’s title plus description (blank line between them). It does not include day, time, room, or speakers. |
| `vector` | Dense embedding of `text`, used for similarity search. |
| `metadata` | Structured fields (see metadata description below) |


| Field | Description |
|--------------|------------|
| `metadata.title` | Title of the session|
| `metadata.day` | Date of the session |
| `metadata.time` | Time slot of the session |
| `metadata.room` | Room where the session takes place |
| `metadata.type` | One of 'keynote', 'workshop', 'talk', 'track_keynote', 'lightning', 'expo_session' |
| `metadata.track` | Track |
| `metadata.speakers` | Name(s) of the speaker(s) as a single comma-separated string |


### Create semantic search tool

Let's create a [retriever tool](https://docs.langchain.com/oss/python/langgraph/agentic-rag) called `find_conference_sessions` that lets the agent run semantic search queries against the Elasticsearch index.

First, we setup an Elasticsearch vector store with a [Jina embedding model](https://docs.langchain.com/oss/python/integrations/embeddings/jina) to generate embeddings at query time.

In [4]:
from langchain_community.embeddings import JinaEmbeddings
from langchain_elasticsearch import ElasticsearchStore

# Set up the embedding model
embeddings = JinaEmbeddings(
    jina_api_key=JINA_API_KEY, 
    model_name="jina-embeddings-v5-text-small"
)

# Set up the vector store
vector_store = ElasticsearchStore(
    es_url=ES_URL,
    es_user=ES_USERNAME,
    es_password=ES_PASSWORD,
    index_name="conference_schedule",
    embedding=embeddings,
)

Convert the Python function to a tool using the [`@tool` decorator](https://reference.langchain.com/python/langchain-core/tools/convert/tool). By default the function name becomes the **tool name** and the docstring becomes the **tool description**.

**Best practices for tool descriptions:**
- Core purpose: A high-level summary of what the tool does
- Trigger: When the tool should be used (and when it should not).
- Action: Which specific data the tool retrieves or modifies, and what type of questions it can answer.
- Limitations: What important limitations and constraints exist, such as specific query languages or formats.
- Relationships with other tools: Does one tool affect another tool, or are there any preconditions?
- Examples: Specific few-shot examples of user queries and how to use the tool for them, such as how to determine the optimal search strategy or when to use which operator.

In [9]:
from langchain.tools import tool

@tool()
def find_conference_sessions(query: str) -> str:
    """Search and return information about the sessions at the AI Engineer Europe 2026 conference.

    Args:
        query: The query to search for.

    Returns:
        A string containing the information about the sessions.
    """
    docs = vector_store.similarity_search(query, k=3)
    return "\n\n".join([f"{doc.metadata['type']} by {doc.metadata['speakers']}: {doc.metadata['title']}\nDescription:{doc.page_content}" for doc in docs])

semantic_search_tool = find_conference_sessions

# Test retriever tool
print(semantic_search_tool.invoke({"query": "regulatory constraints"}))

talk by Bilge Yücel: Engineering AI Systems Under Sovereignty Constraints
Description:Engineering AI Systems Under Sovereignty Constraints

Regulatory and jurisdictional constraints are no longer an edge case in AI system design; they now influence architecture decisions as much as model quality. From European Parliament discussions around prioritising non-US technology in public procurement and building a “Eurostack,” to the launch of the AWS European Sovereign Cloud, sovereignty is becoming a concrete technical requirement.

AI systems increasingly need to operate within defined legal and physical boundaries. For engineers, this raises a practical question: **what changes when your AI system is not allowed to leave a jurisdiction — legally, physically, or operationally?**

This talk frames sovereign AI as an **engineering under constraints** problem, not a political one. We’ll explore how requirements like data localization, auditability, model transparency, and jurisdictional contro

### Create the agent

Now assemble your agent with all the components including the semantic search tool (`find_conference_sessions`).

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[semantic_search_tool],
)

## Run the agent

### Example: Agent uses tool to retrieve context

In [7]:
input_message = {
    "role": "user",
    "content": ("Which sessions discuss handling regulatory constraints?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions discuss handling regulatory constraints?
================================== Ai Message ==================================
Tool Calls:
  find_conference_sessions (call_oFYASfWKDBFRtOUJ7BustV0g)
 Call ID: call_oFYASfWKDBFRtOUJ7BustV0g
  Args:
    query: regulatory constraints compliance regulation handling constraints governance policy AI
================================= Tool Message =================================
Name: find_conference_sessions

talk by Bilge Yücel: Engineering AI Under Sovereignty Constraints
Description:Engineering AI Under Sovereignty Constraints

Regulatory and jurisdictional constraints are no longer an edge case in AI system design; they now influence architecture decisions as much as model quality. From European Parliament discussions around prioritising non-US technology in public procurement and building a “Eurostack,” to the launch of the AWS European Sovereign 

**[Exercise] Try it out with a few different queries to see the behavior:**
- Check if the agent doesn't use the tool to retrieve context when not necessary (e.g., "Hello, I'm so excited to be at the AI Engineer Europe 2026 conference!")
- Does the agent call the tool multiple times when the retrieved information is not relevant
- Does the agent ignore retrieve information that isn't helpful to answer the user query? (e.g., "Where does the AI Engineer Conference take place?")

### Example: Where this fails

In [10]:
input_message = {
    "role": "user",
    "content": ("Which sessions should I visit to learn more about GEPA?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions should I visit to learn more about GEPA?
================================== Ai Message ==================================
Tool Calls:
  find_conference_sessions (call_1vbecI3P09okKUbGFBDmwp1Z)
 Call ID: call_1vbecI3P09okKUbGFBDmwp1Z
  Args:
    query: GEPA
================================= Tool Message =================================
Name: find_conference_sessions

keynote by Omar Sanseviero: Gemma, DeepMind's Family of Open Models
Description:Gemma, DeepMind's Family of Open Models

Google DeepMind’s Gemma family is expanding. Join us for a deep dive into the latest models of the Gemma ecosystem. From vibe fine-tuning to Sovereign AI, you'll learn about the latest model capabilities, how to build high-performance applications, and how to get started with open models.

track_keynote by Ryan Lopopolo: Harness Engineering AMA
Description:Harness Engineering AMA

talk by Dan James: Building 